In [ ]:
CONFIG = {
    "TAMANHO_DATASET": 10000,
    "SEED": 42,
    "PROPORCAO_TESTE": 0.2,
    "EPOCAS_BERT": 3,
    "LR_BERT": 2e-5,
    "BATCH_SIZE": 16,
    "MAX_LEN": 128,
    "MODELO_BERT": "neuralmind/bert-base-portuguese-cased",
}

In [ ]:
import pandas as pd

def carregar_dataset(tamanho: int, seed: int) -> pd.DataFrame:
    """
    Carrega artigos da Wikipedia em português via datasets.

    Args:
        tamanho: número de amostras a carregar
        seed: semente para reprodutibilidade

    Returns:
        DataFrame pandas com coluna 'text' e demais metadados
    """
    try:
        from datasets import load_dataset

        print(f"Carregando {tamanho} artigos da Wikipedia em português...")

        # Usar o dataset omarkamali/wikipedia-monthly com configuração para português
        dataset = load_dataset(
            "omarkamali/wikipedia-monthly",
            "latest.pt", 
            split="train",
            streaming=True
        )

        registros = []
        for i, exemplo in enumerate(dataset):
            if i >= tamanho:
                break
            registros.append(exemplo)

        df = pd.DataFrame(registros)

        colunas_texto = ['text', 'content', 'article', 'body']
        for col in colunas_texto:
            if col in df.columns and col != 'text':
                df = df.rename(columns={col: 'text'})
                break

        if 'text' not in df.columns:
            str_cols = [c for c in df.columns if df[c].dtype == object]
            if str_cols:
                df = df.rename(columns={str_cols[0]: 'text'})
            else:
                raise ValueError("Dataset não contém coluna de texto identificável.")

        df = df.dropna(subset=['text'])
        df = df[df['text'].str.strip() != '']
        df = df.head(tamanho).reset_index(drop=True)

        df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

        print(f"Dataset carregado com sucesso: {len(df)} amostras")
        print(f"Colunas disponíveis: {list(df.columns)}")
        return df

    except ImportError as e:
        raise ImportError(
            f"Erro ao importar datasets: {e}. "
            "Execute '!pip install datasets' e reinicie o kernel."
        )
    except Exception as e:
        raise RuntimeError(
            f"Falha ao carregar o dataset Wikipedia PT: {e}. "
            "Verifique a conexão com a internet e a disponibilidade do dataset."
        )


df = carregar_dataset(CONFIG["TAMANHO_DATASET"], CONFIG["SEED"])
df.head()

Carregando 10000 artigos da Wikipedia em português...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/1425 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/234 [00:00<?, ?it/s]

Dataset carregado com sucesso: 10000 amostras
Colunas disponíveis: ['id', 'url', 'title', 'text']


,id,url,title,text
0,12226,https://pt.wikipedia.org/wiki/União_Democrata-...,União Democrata-Cristã,"A União Democrata-Cristã da Alemanha (, CDU), ..."
1,9594,https://pt.wikipedia.org/wiki/Sivuca,Sivuca,"Severino Dias de Oliveira, mais conhecido como..."
2,4812,https://pt.wikipedia.org/wiki/Samoa_Americana,Samoa Americana,"A Samoa Americana (, ; , ; também Amelika Sāmo..."
3,9715,https://pt.wikipedia.org/wiki/Guerra_Civil_Ing...,Guerra Civil Inglesa,"A Guerra Civil Inglesa, ou Grande Rebelião, fo..."
4,9323,https://pt.wikipedia.org/wiki/18_de_junho,18 de junho,Eventos históricos\n 618 — Li Yuan torna-se o ...


In [ ]:
!pip install -U spacy

In [39]:
!python -m spacy download pt_core_news_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 1.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy

nlp = spacy.load("pt_core_news_lg")

In [ ]:
def extrair_entidades(doc):
    return [
        {
            "entity_text": ent.text,
            "label": ent.label_,
            "start": ent.start_char,
            "end": ent.end_char
        }
        for ent in doc.ents
    ]

=================================================

In [ ]:
batch_size = 100

tokens_all = []
labels_all = []

for inicio in range(0, len(df), batch_size):
    fim = min(inicio + batch_size, len(df))

    textos_lote = df["text"].iloc[inicio:fim].fillna("").astype(str).tolist()
    docs_lote = nlp.pipe(textos_lote, batch_size=4)

    for doc in docs_lote:
        tokens = [token.text for token in doc]

        labels = []
        for token in doc:
            if token.ent_iob_ == "O":
                labels.append("O")
            else:
                labels.append(f"{token.ent_iob_}-{token.ent_type_}")

        tokens_all.append(tokens)
        labels_all.append(labels)

    print(f"Processados {fim} de {len(df)} artigos")

Processados 100 de 10000 artigos
Processados 200 de 10000 artigos
Processados 300 de 10000 artigos
Processados 400 de 10000 artigos
Processados 500 de 10000 artigos
Processados 600 de 10000 artigos
Processados 700 de 10000 artigos
Processados 800 de 10000 artigos
Processados 900 de 10000 artigos
Processados 1000 de 10000 artigos
Processados 1100 de 10000 artigos
Processados 1200 de 10000 artigos
Processados 1300 de 10000 artigos
Processados 1400 de 10000 artigos
Processados 1500 de 10000 artigos
Processados 1600 de 10000 artigos
Processados 1700 de 10000 artigos
Processados 1800 de 10000 artigos
Processados 1900 de 10000 artigos
Processados 2000 de 10000 artigos
Processados 2100 de 10000 artigos
Processados 2200 de 10000 artigos
Processados 2300 de 10000 artigos
Processados 2400 de 10000 artigos
Processados 2500 de 10000 artigos
Processados 2600 de 10000 artigos
Processados 2700 de 10000 artigos
Processados 2800 de 10000 artigos
Processados 2900 de 10000 artigos
Processados 3000 de 100

In [ ]:
df["tokens"] = tokens_all
df["bio_labels"] = labels_all

In [ ]:
df.head()

,id,url,title,text,tokens,bio_labels
0,12226,https://pt.wikipedia.org/wiki/União_Democrata-...,União Democrata-Cristã,"A União Democrata-Cristã da Alemanha (, CDU), ...","[A, União, Democrata-Cristã, da, Alemanha, (, ...","[O, B-ORG, I-ORG, I-ORG, I-ORG, O, O, B-ORG, O..."
1,9594,https://pt.wikipedia.org/wiki/Sivuca,Sivuca,"Severino Dias de Oliveira, mais conhecido como...","[Severino, Dias, de, Oliveira, ,, mais, conhec...","[B-PER, I-PER, I-PER, I-PER, O, O, O, O, B-PER..."
2,4812,https://pt.wikipedia.org/wiki/Samoa_Americana,Samoa Americana,"A Samoa Americana (, ; , ; também Amelika Sāmo...","[A, Samoa, Americana, (, ,, ;, ,, ;, também, A...","[O, B-LOC, I-LOC, O, O, O, O, O, O, B-PER, I-P..."
3,9715,https://pt.wikipedia.org/wiki/Guerra_Civil_Ing...,Guerra Civil Inglesa,"A Guerra Civil Inglesa, ou Grande Rebelião, fo...","[A, Guerra, Civil, Inglesa, ,, ou, Grande, Reb...","[O, B-MISC, I-MISC, I-MISC, O, O, B-MISC, I-MI..."
4,9323,https://pt.wikipedia.org/wiki/18_de_junho,18 de junho,Eventos históricos\n 618 — Li Yuan torna-se o ...,"[Eventos, históricos, \n , 618, —, Li, Yuan, t...","[O, O, O, O, O, B-PER, I-PER, O, O, B-PER, B-P..."


In [ ]:
for i in range(len(df)):
    if len(df.loc[i, "tokens"]) != len(df.loc[i, "bio_labels"]):
        print("Erro na linha", i)
        break

In [ ]:
import re

def filtrar_tokens_regex_like(tokens, labels):
    novos_tokens = []
    novas_labels = []

    for token, label in zip(tokens, labels):
        if re.fullmatch(r"[\w']+", token, re.UNICODE):
            novos_tokens.append(token)
            novas_labels.append(label)

    return novos_tokens, novas_labels

resultado = df.apply(
    lambda row: filtrar_tokens_regex_like(row["tokens"], row["bio_labels"]),
    axis=1
)

df["tokens_regex_like"] = resultado.apply(lambda x: x[0])
df["labels_regex_like"] = resultado.apply(lambda x: x[1])

In [ ]:
df.head()

,id,url,title,text,tokens,bio_labels,tokens_filtrados,labels_filtradas,tokens_regex_like,labels_regex_like
0,12226,https://pt.wikipedia.org/wiki/União_Democrata-...,União Democrata-Cristã,"A União Democrata-Cristã da Alemanha (, CDU), ...","[A, União, Democrata-Cristã, da, Alemanha, (, ...","[O, B-ORG, I-ORG, I-ORG, I-ORG, O, O, B-ORG, O...","[A, União, Democrata-Cristã, da, Alemanha, CDU...","[O, B-ORG, I-ORG, I-ORG, I-ORG, B-ORG, O, O, O...","[A, União, da, Alemanha, CDU, também, simplesm...","[O, B-ORG, I-ORG, I-ORG, B-ORG, O, O, O, O, B-..."
1,9594,https://pt.wikipedia.org/wiki/Sivuca,Sivuca,"Severino Dias de Oliveira, mais conhecido como...","[Severino, Dias, de, Oliveira, ,, mais, conhec...","[B-PER, I-PER, I-PER, I-PER, O, O, O, O, B-PER...","[Severino, Dias, de, Oliveira, mais, conhecido...","[B-PER, I-PER, I-PER, I-PER, O, O, O, B-PER, B...","[Severino, Dias, de, Oliveira, mais, conhecido...","[B-PER, I-PER, I-PER, I-PER, O, O, O, B-PER, B..."
2,4812,https://pt.wikipedia.org/wiki/Samoa_Americana,Samoa Americana,"A Samoa Americana (, ; , ; também Amelika Sāmo...","[A, Samoa, Americana, (, ,, ;, ,, ;, também, A...","[O, B-LOC, I-LOC, O, O, O, O, O, O, B-PER, I-P...","[A, Samoa, Americana, também, Amelika, Sāmoa, ...","[O, B-LOC, I-LOC, O, B-PER, I-PER, O, B-LOC, I...","[A, Samoa, Americana, também, Amelika, Sāmoa, ...","[O, B-LOC, I-LOC, O, B-PER, I-PER, O, B-LOC, I..."
3,9715,https://pt.wikipedia.org/wiki/Guerra_Civil_Ing...,Guerra Civil Inglesa,"A Guerra Civil Inglesa, ou Grande Rebelião, fo...","[A, Guerra, Civil, Inglesa, ,, ou, Grande, Reb...","[O, B-MISC, I-MISC, I-MISC, O, O, B-MISC, I-MI...","[A, Guerra, Civil, Inglesa, ou, Grande, Rebeli...","[O, B-MISC, I-MISC, I-MISC, O, B-MISC, I-MISC,...","[A, Guerra, Civil, Inglesa, ou, Grande, Rebeli...","[O, B-MISC, I-MISC, I-MISC, O, B-MISC, I-MISC,..."
4,9323,https://pt.wikipedia.org/wiki/18_de_junho,18 de junho,Eventos históricos\n 618 — Li Yuan torna-se o ...,"[Eventos, históricos, \n , 618, —, Li, Yuan, t...","[O, O, O, O, O, B-PER, I-PER, O, O, B-PER, B-P...","[Eventos, históricos, 618, Li, Yuan, torna-se,...","[O, O, O, B-PER, I-PER, O, O, B-PER, B-PER, I-...","[Eventos, históricos, 618, Li, Yuan, o, Impera...","[O, O, O, B-PER, I-PER, O, B-PER, B-PER, I-PER..."


In [ ]:
df = df.drop(columns=["tokens_filtrados", "labels_filtradas"])

In [ ]:
df = df.drop(columns=["tokens","bio_labels"])

In [ ]:
df = df.rename(columns={"tokens_regex_like": "tokens"})
df = df.rename(columns={"labels_regex_like": "labels"})

In [ ]:
df.head()

,id,url,title,text,tokens,labels
0,12226,https://pt.wikipedia.org/wiki/União_Democrata-...,União Democrata-Cristã,"A União Democrata-Cristã da Alemanha (, CDU), ...","[A, União, da, Alemanha, CDU, também, simplesm...","[O, B-ORG, I-ORG, I-ORG, B-ORG, O, O, O, O, B-..."
1,9594,https://pt.wikipedia.org/wiki/Sivuca,Sivuca,"Severino Dias de Oliveira, mais conhecido como...","[Severino, Dias, de, Oliveira, mais, conhecido...","[B-PER, I-PER, I-PER, I-PER, O, O, O, B-PER, B..."
2,4812,https://pt.wikipedia.org/wiki/Samoa_Americana,Samoa Americana,"A Samoa Americana (, ; , ; também Amelika Sāmo...","[A, Samoa, Americana, também, Amelika, Sāmoa, ...","[O, B-LOC, I-LOC, O, B-PER, I-PER, O, B-LOC, I..."
3,9715,https://pt.wikipedia.org/wiki/Guerra_Civil_Ing...,Guerra Civil Inglesa,"A Guerra Civil Inglesa, ou Grande Rebelião, fo...","[A, Guerra, Civil, Inglesa, ou, Grande, Rebeli...","[O, B-MISC, I-MISC, I-MISC, O, B-MISC, I-MISC,..."
4,9323,https://pt.wikipedia.org/wiki/18_de_junho,18 de junho,Eventos históricos\n 618 — Li Yuan torna-se o ...,"[Eventos, históricos, 618, Li, Yuan, o, Impera...","[O, O, O, B-PER, I-PER, O, B-PER, B-PER, I-PER..."


In [ ]:
labels_unicas = set()

for lista in df["labels"]:
    labels_unicas.update(lista)

print(labels_unicas)

{'B-LOC', 'I-ORG', 'B-ORG', 'I-LOC', 'B-MISC', 'I-MISC', 'O', 'I-PER', 'B-PER'}


In [33]:
df_labels = df["labels"]

In [34]:
df_labels.head()

,labels
0,"[O, B-ORG, I-ORG, I-ORG, B-ORG, O, O, O, O, B-..."
1,"[B-PER, I-PER, I-PER, I-PER, O, O, O, B-PER, B..."
2,"[O, B-LOC, I-LOC, O, B-PER, I-PER, O, B-LOC, I..."
3,"[O, B-MISC, I-MISC, I-MISC, O, B-MISC, I-MISC,..."
4,"[O, O, O, B-PER, I-PER, O, B-PER, B-PER, I-PER..."


In [37]:
df_labels = df[["labels"]]  # duas colchetes

In [38]:
df_labels.to_parquet("dataset_labels.parquet", index=False)

In [ ]:
df.loc[0, "text"]

'A União Democrata-Cristã da Alemanha (, CDU), também simplesmente chamada de União Democrata-Cristã, é um partido político alemão de centro-direita fundado entre 1945 e 1950.\n\nEm associação com o seu "partido irmão", a União Social-Cristã (CSU), é o segundo maior partido alemão em termos de membros. Federalista, a CDU concorre às eleições em todos os estados federais, com exceção da Baviera, e a CSU, por sua vez, apenas lá. Ambos os partidos formam um grupo parlamentar CDU/CSU (também conhecido como União) no Bundestag.\n\nA CDU foi primeiramente fundada imediatamente após a Segunda Guerra Mundial em 1945 e em uma segunda tentativa na primeira conferência federal do partido em 1950 como um partido cristão não denominacional. Isso a diferenciava do católico Partido do Centro, que personificava os valores democratas-cristãos em toda a República de Weimar. Suas raízes ideológicas são a Doutrina Social da Igreja, o conservadorismo e o ordoliberlismo.Sven-Uwe Schmitz (2009). Konservatism